# Day 6 Lab — Build an Agent on Bedrock: Loop → Strands → AgentCore

**Offline by default.** Every cell runs without AWS until you flip one switch. Fill each `# TODO`.

**Anchor:** TravelMind **UC-A1** (booking-exception handler).
Tools: `lookup_booking` · `get_disruption_reason` · `get_rebooking_options`.

**How to use this notebook**
- Run top to bottom. Cells marked `# TODO` are yours to complete.
- The mock returns canned Bedrock responses, so the *loop logic* runs with no AWS.
- To hit **real Bedrock**: set `USE_REAL_BEDROCK = True` (needs AWS creds + model access), then re-run.
- Deploying to AgentCore is a terminal + console task — the last section writes the file; the **Runbook** deploys it.

## 0 · Setup  *(provided — just run these three cells)*

In [ ]:
USE_REAL_BEDROCK = False          # flip to True to call real Bedrock (needs AWS creds + model access)
REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # us. inference profile (a bare id => ValidationException)
import json

In [ ]:
# Mock tools. The DATA is faked; in real mode the real model still drives the conversation.
def lookup_booking(pnr):        return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}
def get_disruption_reason(pnr): return {"pnr": pnr, "reason": "weather", "detail": "Heavy fog at origin airport"}
def get_rebooking_options(pnr): return {"pnr": pnr, "options": [{"flight": "AI-318", "dep": "18:40"},
                                                                {"flight": "6E-552", "dep": "21:15"}]}
TOOLS = {"lookup_booking": lookup_booking,
         "get_disruption_reason": get_disruption_reason,
         "get_rebooking_options": get_rebooking_options}

In [ ]:
# Offline mock of bedrock-runtime.converse — returns the same shapes real Bedrock does.
class MockBedrockRuntime:
    def converse(self, modelId=None, messages=None, toolConfig=None, system=None, **kw):
        n = sum(1 for m in messages for c in m.get("content", [])
                if isinstance(c, dict) and "toolResult" in c)          # how many tool results so far
        plan = [("lookup_booking",        {"pnr": "JX48Q2"}, "tu_1"),
                ("get_disruption_reason", {"pnr": "JX48Q2"}, "tu_2"),
                ("get_rebooking_options", {"pnr": "JX48Q2"}, "tu_3")]
        if n < len(plan):
            name, inp, tid = plan[n]
            return {"output": {"message": {"role": "assistant",
                        "content": [{"toolUse": {"toolUseId": tid, "name": name, "input": inp}}]}},
                    "stopReason": "tool_use", "usage": {"inputTokens": 120 + n*40, "outputTokens": 30}}
        txt = ("Your flight AI-302 on 2026-06-12 (PNR JX48Q2) was cancelled due to weather "
               "(heavy fog at origin). Two rebooking options: AI-318 at 18:40, or 6E-552 at 21:15. "
               "Want me to hold one?")
        return {"output": {"message": {"role": "assistant", "content": [{"text": txt}]}},
                "stopReason": "end_turn", "usage": {"inputTokens": 260, "outputTokens": 60}}

def get_client():
    if USE_REAL_BEDROCK:
        import boto3
        return boto3.client("bedrock-runtime", region_name=REGION)
    return MockBedrockRuntime()

print("setup ready · mode:", "REAL Bedrock" if USE_REAL_BEDROCK else "offline mock")

## 1 · Define the tools (`toolConfig`)

A tool = **name** + **description** (the routing signal the model reads) + a JSON **input schema**.
`lookup_booking` is done for you. Add the other two.

In [ ]:
def spec(name, desc, props, req):
    return {"toolSpec": {"name": name, "description": desc,
                         "inputSchema": {"json": {"type": "object", "properties": props, "required": req}}}}
PNR = {"pnr": {"type": "string", "description": "6-char PNR code"}}

tool_config = {"tools": [
    spec("lookup_booking", "Look up a booking by its PNR. Use when the user gives a PNR.", PNR, ["pnr"]),
    # TODO: add a spec for get_disruption_reason  ("Why a booking was delayed or cancelled.")
    # TODO: add a spec for get_rebooking_options  ("Alternative flights for a disrupted booking.")
]}

assert len(tool_config["tools"]) == 3, "Add the two missing tool specs"
print("tools:", [t["toolSpec"]["name"] for t in tool_config["tools"]])

## 2 · One tool turn, by hand

Send the first request. The model replies with `stopReason='tool_use'` and a `toolUse` block — it is **asking** you to run a tool, not running it itself.

In [ ]:
SYSTEM = "You are TravelMind, a booking-exception assistant. Never invent a PNR."
rt = get_client()
messages = [{"role": "user", "content": [{"text": "Status of PNR JX48Q2 and my options?"}]}]
resp = rt.converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages, toolConfig=tool_config)

print("stopReason:", resp["stopReason"])
tu = resp["output"]["message"]["content"][-1]["toolUse"]
print("tool:", tu["name"], "| input:", tu["input"], "| id:", tu["toolUseId"])

### The step everyone forgets

Before you send the result back, **append the assistant's tool_use message**. Then run the tool and return the result **in a user turn**, echoing the same `toolUseId`.

In [ ]:
# TODO (1): append the assistant's tool_use message to `messages` FIRST
#           (without this, Bedrock rejects the toolResult)

out = TOOLS[tu["name"]](**tu["input"])         # run the tool

# TODO (2): append a USER turn carrying the toolResult.
#           It must include toolUseId = tu["toolUseId"] and content [{"json": out}]

resp = rt.converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages, toolConfig=tool_config)
print("next stopReason:", resp["stopReason"])

## 3 · The loop (with a guard)

A real run chains several tools. Loop until `end_turn`, and cap the steps so a misbehaving model can't spin forever (this guard is what prevents the hackathon timeout).

In [ ]:
def run_agent(user_text, max_steps=6, verbose=True):
    rt = get_client()
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    resp = rt.converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages, toolConfig=tool_config)
    steps = 0
    # TODO: loop while resp["stopReason"] == "tool_use" AND steps < max_steps
    #   - steps += 1
    #   - tu = resp["output"]["message"]["content"][-1]["toolUse"]
    #   - messages.append(resp["output"]["message"])            # assistant turn first
    #   - dispatch: out = TOOLS[tu["name"]](**tu["input"])      # wrap in try/except -> {"error": str(e)}
    #   - append the toolResult (user turn, matching id)
    #   - resp = rt.converse(... messages ...)                  # go again
    return resp["output"]["message"]["content"][0]["text"]

print(run_agent("My flight on JX48Q2 was disrupted — what happened and what are my options?"))

In [ ]:
# self-check
try:
    _ans = run_agent("My flight on JX48Q2 was disrupted — what happened and what are my options?", verbose=False)
    assert "AI-318" in _ans and "weather" in _ans
    print("loop works ✓")
except Exception as e:
    print("not complete yet:", repr(e))

### Stop conditions to recognise
`end_turn` = done · `max_tokens` = ran out of room (raise the cap) · `tool_use` = wants another tool.
The guard `steps < max_steps` is your safety net.

## 4 · Strands — let the framework write the loop

Same agent, ~12 lines. `@tool` turns a function into a tool (docstring = description, type hints = schema).
Strands runs the loop you just wrote — against **real Bedrock**, so this cell executes only in real mode. Offline, read it and compare line counts with Section 3.

In [ ]:
# Runs only against real Bedrock. Offline, just read the code.
if USE_REAL_BEDROCK:
    from strands import Agent, tool
    from strands.models import BedrockModel

    # TODO: decorate each tool with @tool and a one-line docstring
    #       (lookup_booking, get_disruption_reason, get_rebooking_options)

    model = BedrockModel(model_id=MODEL, region_name=REGION)
    # TODO: agent = Agent(model=model, tools=[...]) ; then print(agent("Status of PNR JX48Q2 and my options?"))
else:
    print("Strands needs real Bedrock. Set USE_REAL_BEDROCK=True and `pip install strands-agents` to run this.")

## 5 · Package for AgentCore (write the deploy file)

Strands solved the *code*. AgentCore runs it in production. **Three additions** wrap your agent as a service (port 8080, `POST /invocations`, `GET /ping`). This cell writes `travelmind_agent.py` and `requirements.txt` to disk — you deploy them with the **Runbook**.

In [ ]:
agent_src = '''# travelmind_agent.py  — deploy with the Runbook (agentcore configure / launch / invoke)
from bedrock_agentcore import BedrockAgentCoreApp          # +++ AgentCore wrapper
from strands import Agent, tool
from strands.models import BedrockModel

MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"     # us. inference profile

@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its PNR code."""
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}

@tool
def get_disruption_reason(pnr: str) -> dict:
    """Why a booking was delayed or cancelled."""
    return {"pnr": pnr, "reason": "weather", "detail": "Heavy fog at origin airport"}

@tool
def get_rebooking_options(pnr: str) -> list:
    """Alternative flights for a disrupted booking."""
    return [{"flight": "AI-318", "dep": "18:40"}, {"flight": "6E-552", "dep": "21:15"}]

model = BedrockModel(model_id=MODEL, region_name="us-west-2")
agent = Agent(model=model,
              tools=[lookup_booking, get_disruption_reason, get_rebooking_options],
              system_prompt="You are TravelMind, a booking-exception assistant. Never invent a PNR.")

app = BedrockAgentCoreApp()                                # +++ create the app

@app.entrypoint                                            # +++ mark the entrypoint
def invoke(payload):
    result = agent(payload.get("prompt", ""))
    return {"result": str(result)}

if __name__ == "__main__":
    app.run()                                              # +++ serve on :8080  (/invocations, /ping)
'''
with open("travelmind_agent.py", "w") as f:
    f.write(agent_src)
with open("requirements.txt", "w") as f:
    f.write("bedrock-agentcore\nstrands-agents\n")
print("wrote travelmind_agent.py and requirements.txt")

In [ ]:
print("Now, in a terminal (the Runbook has the click-by-click + expected output):\n")
print("  agentcore configure -e travelmind_agent.py --disable-memory")
print("  agentcore launch")
print("  agentcore invoke '{\"prompt\": \"Status of PNR JX48Q2 and my options?\"}'")

## 6 · Recap — the anti-patterns that bite

- Append the **assistant tool_use message** before the `toolResult`
- **Match the `toolUseId`** between `toolUse` and `toolResult`
- **Guard the loop** (`steps < max_steps`)
- **`bedrock-agent-runtime`** for Knowledge Base retrieve — not `bedrock-runtime`
- **`us.` / `global.` inference profile** on the model id
- Don't mix the **two `agentcore` CLIs** (Python starter toolkit vs the npm `@aws/agentcore`)
- **Stream long runs** or they time out

Deploy steps and failure→fix live in the **Live-AWS Runbook**.